# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [9]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
df.shape
df.columns.tolist()
df.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

## A.2. Missing values & Duplicate data

In [3]:
df.isna().sum()
df.duplicated().sum()

np.int64(5268)

## A.3. Invalid values

In [4]:
df[(df['Quantity'] <= 0) | (df['UnitPrice'] <= 0)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [10]:
raw_df = df.copy()
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()
df['Sales'] = df['Quantity'] * df['UnitPrice']

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
numeric_df = df[['Quantity', 'UnitPrice', 'Sales']]
pd.DataFrame({'mean': numeric_df.mean(), 'median': numeric_df.median(), 'mode': numeric_df.mode().iloc[0]})

,mean,median,mode
Quantity,10.542037,3.00,1.00
UnitPrice,3.907625,2.08,1.25
Sales,20.121871,9.90,15.00


## Group 2 — Dispersion

In [7]:
pd.DataFrame({'range': numeric_df.max() - numeric_df.min(), 'variance': numeric_df.var(), 'std': numeric_df.std(), 'IQR': numeric_df.quantile(0.75) - numeric_df.quantile(0.25)})

,range,variance,std,IQR
Quantity,80994.000,24187.752994,155.524124,9.00
UnitPrice,13541.329,1289.936149,35.915681,2.88
Sales,168469.599,73092.768604,270.356743,13.95


## Group 3 — Location and Shape

In [8]:
pd.DataFrame({'Q1': numeric_df.quantile(0.25), 'median': numeric_df.median(), 'Q3': numeric_df.quantile(0.75), 'skewness': numeric_df.skew(), 'kurtosis': numeric_df.kurt()})

,Q1,median,Q3,skewness,kurtosis
Quantity,1.00,3.00,10.00,471.727716,236462.342826
UnitPrice,1.25,2.08,4.13,206.087555,62483.142715
Sales,3.75,9.90,17.70,506.706012,297651.661046


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [11]:
country_sales = df.groupby('Country')['Sales'].sum().sort_values(ascending=False)
pd.DataFrame({'Sales': country_sales, 'percentage': country_sales / country_sales.sum() * 100})

,Sales,percentage
Country,,
United Kingdom,9025222.084,84.611315
Netherlands,285446.340,2.676055
EIRE,283453.960,2.657376
Germany,228867.140,2.145626
France,209715.110,1.966076
Australia,138521.310,1.298635
Spain,61577.110,0.577284
Switzerland,57089.900,0.535217
Belgium,41196.340,0.386215


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [12]:
df.groupby('Description')['Sales'].sum().sort_values(ascending=False).head(10)

Description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174484.74
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106292.77
PARTY BUNTING                          99504.33
JUMBO BAG RED RETROSPOT                94340.05
MEDIUM CERAMIC TOP STORAGE JAR         81700.92
Manual                                 78112.82
POSTAGE                                78101.88
RABBIT NIGHT LIGHT                     66964.99
Name: Sales, dtype: float64

## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [13]:
monthly_sales = df.groupby(df['InvoiceDate'].dt.to_period('M'))['Sales'].sum()
monthly_sales

InvoiceDate
2010-12     823746.140
2011-01     691364.560
2011-02     523631.890
2011-03     717639.360
2011-04     537808.621
2011-05     770536.020
2011-06     761739.900
2011-07     719221.191
2011-08     759138.380
2011-09    1058590.172
2011-10    1154979.300
2011-11    1509496.330
2011-12     638792.680
Freq: M, Name: Sales, dtype: float64

## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [14]:
order_sales = df.groupby(['InvoiceNo', 'Country'])['Sales'].sum()
order_sales.groupby('Country').mean().sort_values(ascending=False).rename('AOV')

Country
Singapore               3039.898571
Netherlands             3036.663191
Australia               2430.198421
Japan                   1969.282632
Lebanon                 1693.880000
Hong Kong               1426.527273
Brazil                  1143.600000
Sweden                  1066.064722
Switzerland             1057.220370
Denmark                 1053.074444
Israel                  1016.907500
Norway                  1004.595556
RSA                     1002.310000
EIRE                     984.215139
Greece                   952.104000
Cyprus                   849.398750
Channel Islands          786.555385
USA                      716.078000
Spain                    684.190111
United Arab Emirates     634.093333
Iceland                  615.714286
Canada                   611.063333
Austria                  599.922353
Portugal                 581.846552
Finland                  549.904390
Malta                    545.118000
France                   534.987526
United Kingdom      

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [15]:
(raw_df.assign(return_cancelled=raw_df['Quantity'] < 0)
 .groupby('Country')['return_cancelled']
 .mean()
 .mul(100)
 .sort_values(ascending=False)
 .rename('return_cancelled_percentage'))

Country
USA                     38.487973
Czech Republic          16.666667
Malta                   11.811024
Japan                   10.335196
Saudi Arabia            10.000000
Australia                5.877681
Italy                    5.603985
Bahrain                  5.263158
Germany                  4.770932
EIRE                     3.684724
Poland                   3.225806
Singapore                3.056769
Sweden                   2.380952
Denmark                  2.313625
Spain                    1.894986
United Kingdom           1.855178
Belgium                  1.836636
Switzerland              1.748252
France                   1.741264
European Community       1.639344
Finland                  1.438849
Hong Kong                1.388889
Channel Islands          1.319261
Norway                   1.289134
Cyprus                   1.286174
Portugal                 1.184990
Austria                  0.748130
Greece                   0.684932
Israel                   0.673401
Nether

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

United Kingdom là thị trường chủ lực, đóng góp khoảng 84,61% tổng doanh thu. Sản phẩm có doanh thu cao nhất là DOTCOM POSTAGE, tiếp theo là REGENCY CAKESTAND 3 TIER và PAPER CRAFT, LITTLE BIRDIE. Doanh số có tính mùa vụ rõ rệt, tăng mạnh từ tháng 9 đến tháng 11 và đạt đỉnh vào tháng 11. Giá trị đơn hàng trung bình cao nhất thuộc về Singapore, nhưng cần thận trọng khi so sánh vì số lượng đơn hàng giữa các quốc gia khác nhau. Tỷ lệ giao dịch có Quantity âm cao nhất ở USA, cho thấy cần theo dõi riêng các giao dịch trả hàng hoặc hủy đơn tại thị trường này.